# AMS-SkipGNN Kaggle T4 runner

Use **GPU T4**, Internet **ON**, and **Save & Commit All**.

This notebook clones `aryonmt/finalProject`, fetches SkipGNN fold-1 splits, runs smoke tests, then training. The last cell writes **one** zip:

`/kaggle/working/ams_skipgnn_kaggle_bundle.zip`

Download only that file. Drop it at the repo root so the artifacts can be unpacked into `results/`, `figures/`, and `notebooks/`.


In [ ]:
import os, sys, platform, subprocess, shutil
from pathlib import Path

print('python', sys.version)
print('platform', platform.platform())
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu', torch.cuda.get_device_name(0))
except Exception as e:
    print('torch import failed', e)

REPO = 'https://github.com/aryonmt/finalProject.git'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
ROOT = WORK / 'finalProject'
if (Path.cwd() / 'src' / 'models').exists():
    ROOT = Path.cwd()
    print('already in repo', ROOT)
else:
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO, str(ROOT)])
    print('cloned', ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('cwd', os.getcwd())


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'])
print('pip install -e . done')
subprocess.check_call([sys.executable, 'scripts/fetch_data.py'])
print('data fetch done')


In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pytest', '-q', 'tests/test_smoke.py'])
print('pytest rc', rc)
assert rc == 0, 'smoke tests failed'


In [ ]:
import os, subprocess, sys, time
stage = os.environ.get('STAGE', '1')
print('STAGE', stage, '(0=smoke --quick, 1=DTI+DDI full, 2=full suite)')
t0 = time.time()
datasets = ['DTI'] if stage == '0' else ['DTI', 'DDI']
for ds in datasets:
    cmd = [sys.executable, 'scripts/run_benchmark.py', '--dataset', ds, '--models', 'gcn', 'skipgnn', 'ams', 'heuristic', '--device', 'auto']
    if stage == '0':
        cmd += ['--quick']
    print('running', cmd)
    subprocess.check_call(cmd)
print('DTI/DDI minutes', round((time.time()-t0)/60, 2))
print('checkpoint: results/ now has DTI/DDI CSVs; download before STAGE=2 if the session might die')


In [ ]:
import os, subprocess, sys
stage = os.environ.get('STAGE', '1')
if stage in {'0', '1'}:
    print('skipping ablation/robustness/PPI/GDI in STAGE=%s; set STAGE=2 for the rest' % stage)
else:
    subprocess.check_call([sys.executable, 'scripts/run_ablation.py', '--dataset', 'DTI'])
    subprocess.check_call([sys.executable, 'scripts/run_robustness.py', '--dataset', 'DTI'])
    subprocess.check_call([sys.executable, 'scripts/run_benchmark.py', '--dataset', 'PPI', '--models', 'skipgnn', 'ams'])
    subprocess.check_call([sys.executable, 'scripts/run_benchmark.py', '--dataset', 'GDI', '--models', 'skipgnn', 'ams'])
print('stage extras done')


In [ ]:
import json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path

subprocess.call([sys.executable, 'scripts/make_figures.py'])

repo = Path.cwd()
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else repo
staging = work / '_kaggle_bundle_staging'
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)

def copy_tree(src: Path, dest: Path) -> int:
    if not src.exists():
        return 0
    n = 0
    dest.mkdir(parents=True, exist_ok=True)
    for path in src.rglob('*'):
        if path.is_file() and path.name not in {'.gitkeep', '.DS_Store'}:
            target = dest / path.relative_to(src)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
            n += 1
    return n

n_results = copy_tree(repo / 'results', staging / 'results')
n_figures = copy_tree(repo / 'figures', staging / 'figures')

nb_candidates = [
    Path('/kaggle/working/__notebook__.ipynb'),
    Path('/kaggle/working/__notebook_source__.ipynb'),
    work / 'ams_skipgnn_kaggle_runner.ipynb',
    repo / 'notebooks' / 'ams_skipgnn_kaggle_runner.ipynb',
]
nb_src = next((p for p in nb_candidates if p.is_file()), None)
if nb_src is not None:
    dest_nb = staging / 'notebooks' / 'ams_skipgnn_kaggle_runner.ipynb'
    dest_nb.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(nb_src, dest_nb)

files = sorted(p.relative_to(staging).as_posix() for p in staging.rglob('*') if p.is_file())
manifest = {
    'created_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'stage': os.environ.get('STAGE', '1'),
    'cwd': str(repo),
    'n_result_files': n_results,
    'n_figure_files': n_figures,
    'notebook_source': str(nb_src) if nb_src else None,
    'files': files,
    'import_map': {
        'results/': 'results/',
        'figures/': 'figures/',
        'notebooks/ams_skipgnn_kaggle_runner.ipynb': 'notebooks/ams_skipgnn_kaggle_runner.ipynb',
    },
}
try:
    import torch
    manifest['torch'] = torch.__version__
    manifest['cuda'] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        manifest['gpu'] = torch.cuda.get_device_name(0)
except Exception:
    pass
(staging / 'MANIFEST.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(staging / 'IMPORT.txt').write_text(
    'Drop this zip at the repo root. Unpack results/, figures/, and notebooks/ over the repo.\n',
    encoding='utf-8',
)

zip_path = work / 'ams_skipgnn_kaggle_bundle.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in staging.rglob('*'):
        if path.is_file():
            zf.write(path, path.relative_to(staging).as_posix())
shutil.rmtree(staging, ignore_errors=True)

print('DOWNLOAD THIS FILE:', zip_path)
print('bytes', zip_path.stat().st_size)
print('files', files)
print('KAGGLE RUN COMPLETE — download only ams_skipgnn_kaggle_bundle.zip')
